## 在生成 处理后的库存 后 和 生成 在途货件 前 处理好后运行

In [3]:
import pandas as pd
time = '20261109'

In [4]:
# 读取新店铺SKU信息的销占比表
sales_new_df = pd.read_excel('E:\sontu\shipping_recommendations\src_data\Listing销占比程序表-模板-CA.xlsx')
sales_new_df['id：站点SKU'] = sales_new_df['站点'].astype(str) + sales_new_df['积加SKU'].astype(str)
# 选取'店铺-站点'以HK_开头的行
sales_new_df = sales_new_df[sales_new_df['店铺-站点'].str.startswith('HK_')]
# 重命名sales_new_df的列名
sales_new_df.rename(columns={'MSKU': 'MSKU_new', '店铺-站点': '店铺-站点_new'}, inplace=True)

# 读取需要修改SKU信息的表
# 1.读取老店铺SKU信息的销占比表
# sales_old_df = pd.read_excel('E:\sontu\shipping_recommendations\src_data\Listing销占比程序表-模板-CA.xlsx')
# 2.读取处理后的库存表
inventory_df = pd.read_excel(f'E:\sontu\shipping_recommendations\src_data\处理后的库存\库存{time}.xlsx')
# 3.读取发货单清单表的详情表
# shipment_df = pd.read_excel(f'E:\sontu\shipping_recommendations\src_data\在途货件\发货单清单{time}.xlsx', sheet_name='system_发货单')
shipment_details_df = pd.read_excel(f'E:\sontu\shipping_recommendations\src_data\在途货件\发货单清单{time}.xlsx', sheet_name='system_发货单明细')



d:\Programs\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
# # 开始修改表内信息
# # 1. 先修改销占比表 —— 销占比表的修改可能不需要，之后运营提交的是修改过来的版本
# sales_old_df['id：站点SKU'] = sales_old_df['站点'].astype(str) + sales_old_df['积加SKU'].astype(str)

# sales_new_merge = pd.merge(sales_new_df[['id：站点SKU','MSKU_new', '店铺-站点_new']], sales_old_df, on='id：站点SKU', how='right')
# # 当MSKU_new不为空时，替换MSKU和店铺-站点的值
# # sales_new_merge['MSKU'] = sales_new_merge.apply(lambda row: row['MSKU_new'] if pd.notnull(row['MSKU_new']) else row['MSKU'], axis=1)
# # sales_new_merge['店铺-站点'] = sales_new_merge.apply(lambda row: row['店铺-站点_new'] if pd.notnull(row['店铺-站点_new']) else row['店铺-站点'], axis=1)
# sales_new_merge['MSKU'] = sales_new_merge['MSKU_new'].combine_first(sales_new_merge['MSKU'])
# sales_new_merge['店铺-站点'] = sales_new_merge['店铺-站点_new'].combine_first(sales_new_merge['店铺-站点'])

# # 删除不再需要的列
# sales_new_merge.drop(columns=['id：站点SKU', 'MSKU_new', '店铺-站点_new'], inplace=True)
# sales_new_merge.to_excel(f'E:\sontu\shipping_recommendations\src_data\Listing销占比程序表-模板-CA-修改后.xlsx', index=False)


In [4]:
# inventory_df.columns

In [5]:
# 2.修改库存表
inventory_df['id：站点SKU'] = inventory_df['站点'].astype(str) + inventory_df['SKU'].astype(str)
inventory_new_df = pd.merge(inventory_df, sales_new_df[['id：站点SKU','MSKU_new', '店铺-站点_new']], on='id：站点SKU', how='left')

inventory_new_df['MSKU'] = inventory_new_df['MSKU_new'].combine_first(inventory_new_df['MSKU'])
inventory_new_df['仓库'] = inventory_new_df['店铺-站点_new'].combine_first(inventory_new_df['仓库'])

# 选取 'MSKU_new' 为空的行
inventory_new_null_df = inventory_new_df[inventory_new_df['MSKU_new'].isnull()]
inventory_new_null_df.drop(columns=['id：站点SKU', 'MSKU_new', '店铺-站点_new'], inplace=True)

# 选取 'MSKU_new' 不为空的行
inventory_new_notnull_df = inventory_new_df[inventory_new_df['MSKU_new'].notnull()]
# 删除不再需要的列
inventory_new_notnull_df.drop(columns=['id：站点SKU', 'MSKU_new', '店铺-站点_new'], inplace=True)
# 分组求和，数据透视操作
inventory_new_notnull_df = inventory_new_notnull_df.groupby(['站点', '仓库', 'SKU', 'MSKU'], as_index=False).agg({
    '预占总数': 'sum',
    '快递_预占': 'sum',
    '空运_预占': 'sum',
    '海运_预占': 'sum',
    'FBA自提物流_预占': 'sum',
    'FBA预占数': 'sum',
    '已发货(数量)': 'sum',
    '正在接收(数量)': 'sum',
    'FBA在途': 'sum',
    '预留-运营中心转运': 'sum',
    '预留-运营中心正在处理': 'sum',
    'FBA可售': 'sum',
    'FBA可用库存': 'sum',
    '长期仓储库存': 'sum',
    '本地锁仓数': 'sum',
    '本地-共享': 'sum',
    '已下单数量': 'sum',
    '本地-在途': 'sum',
    '已生产未发货': 'sum',
    '7天日均': 'sum',
    '15天日均': 'sum',
    '30天日均': 'sum'
})
# 将为空的行与聚合后不为空的行合并
inventory_new_result = pd.concat([inventory_new_null_df, inventory_new_notnull_df], ignore_index=True)
inventory_new_result.to_excel(f'E:\sontu\shipping_recommendations\src_data\处理后的库存\库存{time}-修改后.xlsx', index=False)

C:\Users\gongy\AppData\Local\Temp\ipykernel_29716\1930617916.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inventory_new_null_df.drop(columns=['id：站点SKU', 'MSKU_new', '店铺-站点_new'], inplace=True)
C:\Users\gongy\AppData\Local\Temp\ipykernel_29716\1930617916.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inventory_new_notnull_df.drop(columns=['id：站点SKU', 'MSKU_new', '店铺-站点_new'], inplace=True)


In [4]:
inventory_new_result.columns

Index(['站点', '仓库', 'SKU', 'MSKU', '预占总数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占', 'FBA预占数', '已发货(数量)', '正在接收(数量)', 'FBA在途', '预留-运营中心转运',
       '预留-运营中心正在处理', 'FBA可售', 'FBA可用库存', '长期仓储库存', '本地锁仓数', '本地-共享', '已下单数量',
       '本地-在途', '已生产未发货', '7天日均', '15天日均', '30天日均'],
      dtype='object')

In [5]:
len(shipment_details_df)

27773

In [6]:
# 3.修改发货单清单表
# 站点需要从仓库列获取，例 如仓库为SEEKWAY:JP_FBA，则站点为JP
shipment_details_df['站点'] = shipment_details_df['仓库'].apply(lambda x: x.split(':')[1].split('_')[0] if pd.notnull(x) and ':' in x else None)
shipment_details_df['id：站点SKU'] = shipment_details_df['站点'].astype(str) + shipment_details_df['SKU'].astype(str)
# 删除站点列
shipment_details_df.drop(columns=['站点'], inplace=True)
sales_new_df['店铺-站点_new'] = sales_new_df['店铺-站点_new'].astype(str) + '_FBA'

shipment_details_new_df = pd.merge(shipment_details_df, sales_new_df[['id：站点SKU','MSKU_new', '店铺-站点_new']], on='id：站点SKU', how='left')
shipment_details_new_df['MSKU'] = shipment_details_new_df['MSKU_new'].combine_first(shipment_details_new_df['MSKU'])
shipment_details_new_df['仓库'] = shipment_details_new_df['店铺-站点_new'].combine_first(shipment_details_new_df['仓库'])
# 删除不再需要的列
shipment_details_new_df.drop(columns=['id：站点SKU','MSKU_new', '店铺-站点_new'], inplace=True)
shipment_details_new_df.to_excel(f'E:\sontu\shipping_recommendations\src_data\在途货件\发货单清单{time}-详情表修改后.xlsx', index=False)

In [7]:
shipment_details_new_df.columns

Index(['发货单号', '关联来源单据', '采购订单', '自定义采购订单', '采购主体', '供应商', '平台站点',
       'ShipmentId', '仓库', '产品名称', 'SKU', 'MSKU', 'ASIN', 'FNSKU', '产品类型',
       '发货量', '已收量', '差异量', '赔偿量', '认丢量', '调整量', '额外收入量', '已处理差异量',
       '单品实物重(KG)', '单品体积重(KG)', '单品总重量', '采购币种', '单品采购单价', '单品尺寸（cm）',
       '包装尺寸（cm）', '箱数', '单箱数量', '单箱净重', '单箱毛重（kg）', '总净重', '总毛重', '单品体积',
       '单品总体积', '外箱尺寸', '总体积', '提货日期', '预计出运日期', '预计到仓日期', '关税币种', '关税分摊单价',
       '关税分摊单价(尾数)', '关税分摊总价', '运杂费币种', '运杂费分摊单价', '运杂费分摊单价(尾数)', '运杂费分摊总价',
       '分摊总价币种', '分摊总价'],
      dtype='object')

In [10]:
len(shipment_details_new_df)

27776